In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import librosa


In [2]:
AUDIO_INPUT=r"C:\Users\Hype G12\OneDrive\Desktop\sintesis"
ADUIO_OUTPUT=r"C:\Users\Hype G12\OneDrive\Desktop\Sintesis_convert"

In [ ]:
SAMPLE_RATE= 16000
EXT= ".wav"


for audio in os.listdir(AUDIO_INPUT):
    if not audio.lower().endswith(EXT):
        continue
    audio_path= os.path.join(AUDIO_INPUT, audio)
    y,  sr= librosa.load(audio_path, sr=SAMPLE_RATE)


    mel_spectogram= librosa.feature.melspectrogram(
        y=y,
        sr= SAMPLE_RATE,
        n_mels= 80,
        hop_length=160,
        n_fft= 400
    )

    mel_db= librosa.power_to_db(mel_spectogram)

    mfcc= librosa.feature.mfcc(
        y=y,
        sr= SAMPLE_RATE,
        n_mfcc= 80,
        hop_length=160,
        n_fft= 400
    )

    cent= librosa.feature.spectral_centroid(
        y=y,
        sr= SAMPLE_RATE,
        n_fft= 400,
        hop_length=160
    )

    f0, voice_flag, voice_prob = librosa.pyin(
        y,
        fmin= librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr,
        hop_length=160
    )

    f0 = np.nan_to_num(f0)

    min_time_frame= min(
        mel_db.shape[1], 
        mfcc.shape[1],
        cent.shape[1], 
        f0.shape[0]
        )
    
    mel_db_trimmed= mel_db[:, :min_time_frame]
    mfcc_trimmed= mfcc[:, :min_time_frame] 
    cent_trimed= cent[:, :min_time_frame]

    f0_trimmed= f0[:min_time_frame]
    f0_trimmed_2d= f0_trimmed[np.newaxis, :]

    combined= np.vstack([mel_db_trimmed, mfcc_trimmed, cent_trimed, f0_trimmed_2d])
    combined= combined[...,np.newaxis]

    os.makedirs(ADUIO_OUTPUT, exist_ok=True)

    fig, ax =plt.subplots(5,1, figsize=(8,10))


    #Waveform
    librosa.display.waveshow(y, sr=sr, ax=ax[0])
    ax[0].set_title("Waveform")

    # Mel-Spectogram
    img_Spectogram=librosa.display.specshow(
        mel_db_trimmed,
        sr=sr,
        hop_length=160,
        x_axis="time",
        y_axis="mel",
        ax=ax[1]
    )
    ax[1].set_title("Mel-Spectogram")
    fig.colorbar(img_Spectogram, ax=ax[1], format="%+2.0f dB")

    #f0
    ax[2].plot(f0_trimmed)
    ax[2].set_title("Pitch (F0)")
    ax[2].set_xlabel("Frame")

    #MFCC
    ax[3].set_title("MFCC")
    img_MFCC=librosa.display.specshow(
        mfcc_trimmed,
        sr=sr,
        hop_length=160,
        x_axis="time",
        ax=ax[3]
    )
    fig.colorbar(img_MFCC, ax=ax[3])


    #SPECTRAL CENTROID
    ax[4].plot(cent_trimed[0], color="orange")
    ax[4].set_title("Spectral Centroid")
    ax[4].set_xlabel("Frame")
     

    save_path=os.path.join(ADUIO_OUTPUT, audio.replace(".wav", "_features.png"))
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)

